# First steps with Spark 

## Data importation

Spark's main object class is the DataFrame, which is a distributed table.
It is analogous to R's or Python (Pandas)'s data frames:

- one row represents an observation,
- one column represents a variable.

But contrary to R or Python, Spark's DataFrames can be distributed over hundred of nodes.

Spark support multiple data formats, and multiple  ways to load them.

- data format : csv, json, parquet (an open source column oriented format)
- can read archive files
- schema detection or user defined schema. For static data, like a json file, schema detection can be use with good results.

Spark has multiple syntaxes to import data. Some are simple with no customisation, others are more complexes but you can specify options.

The simplest syntaxes to load a json or a csv file are :

```{.python}
# JSON
json_df = spark.read.json([location of the file])
# csv
csv_df = spark.read.csv([location of the file])

```

In the future, you may consult the [Data Source documentation](https://spark.apache.org/docs/latest/sql-data-sources.html) to have the complete description of Spark's reading abilities.

The data you will use in this lab are real data from the twitter/X [sampled stream API](https://developer.twitter.com/en/docs/twitter-api/tweets/sampled-stream/introduction) and [filtered stream API](https://developer.twitter.com/en/docs/twitter-api/tweets/filtered-stream/introduction). Since renamed X https://docs.x.com/x-api/introduction. The tweets folder contains more than 50 files and more than 2 million tweets. The tweets was collected between the 14/04/2021 and the 18/04/2021. The total collection time was less than 10 hours.


⚙️ The file we will work on is a `JSONL` (JSON-line) format, which means that each line of it is a JSON object. A JSON object is just a Python dictionary or a JavaScript object and looks like this: `{ key1: value1, key2: ["array", "of", "many values]}`. This file has been compressed into a `GZ` archive, hence the `.jsonl.gz` ending. Also this file is not magically appearing in your S3 storage. It is hosted on one of your teacher's bucket and has been made public, so that you can access it.

It's possible to load multiple file in a unique DataFrame. It's useful when you have daily files and want to process them all. It's the same syntax as the previous one, just specify a folder.

### Create a spark session



### Import the data

Load the json file stored here : 
  - `s3a://ludo2ne/diffusion/tweets.jsonl.gz`
and give your dataframe a significant name like `df_tweet`



Use function `cache()` on the data frame. Caching is a performance optimization technique that allows you to persist an intermediate or final result of a computation in memory, reducing the need to recompute the data when it is accessed multiple time

df.cache()

## Is Spark lazy ? DataFrame basic manipulations



If DataFrames are immutable, they can however be **_transformed_** in other DataFrames, in the sense that a modified copy is returned.
Such **transformations** include: filtering, sampling, dropping columns, selecting columns, adding new columns...

First, you can get information about the columns with:

```{.python}
df.columns       # get the column names
df.schema        # get the column names and their respective type
df.printSchema() # same, but human-readable
```

You can select columns with the `select()` method. It takes as argument a list of column name. For example :

```{.python}
df_with_less_columns = df\
  .select("variable3","variable_four","variable-6")

# Yes, you do need the ugly \ at the end of the line,
# if you want to chain methods between lines in Python
```

You can get nested columns easily with :

```{.python}
df.select("parentField.nestedField")
```

To filter data you could use the `filter()` method. It take as input an expression that gets evaluated for each observation and should return a boolean. Sampling is performed with the `sample()` method. For example :

```{.python}
df_with_less_rows = df\
  .sample(fraction=0.001)\
  .filter(df.variable1=="value")\
  .show(10)
```

As said before your data are distributed over multiple nodes (executors) and data inside a node are split into partitions. Then each transformations will be run in parallel. They are called *narrow transformation* For example, to sample a DataFrame, Spark sample every partitions in parallel because sample all partition produce the sample DataFrame. For some transformations, like `groupBy()` it's impossible, and it's cannot be run in parallel.

![](img/spark_exemple2_pipeline.png)


### Lazy evaluation



This is because Spark has what is known as **lazy evaluation**, in the sense that it will wait as much as it can before performing the actual computation. Said otherwise, when you run an instruction such as:

```{.python}
tweet_author_hashtags = df_tweet.select("auteur","hashtags")
```

... you are not executing anything! Rather, you are building an **execution plan**, to be realised later.

Spark is quite extreme in its laziness, since only a handful of methods called **actions**, by opposition to **transformations**, will trigger an execution. The most notable are:

1. `collect()`, explicitly asking Spark to fetch the resulting rows instead of to lazily wait for more instructions,
2. `take(n)`, asking for `n` first rows
3. `first()`, an alias for `take(1)`
4. `show()` and `show(n)`, human-friendly alternatives
5. `count()`, asking for the numbers of rows
6. all the "write" methods (write on file, write to database), see [here](https://spark.apache.org/docs/3.1.1/api/python/reference/pyspark.sql.html#input-and-output) for the list

**This has advantages:** on huge data, you don't want to accidently perform a computation that is not needed. Also, Spark can optimize each **stage** of the execution in regard to what comes next. For instance, filters will be executed as early as possible, since it diminishes the number of rows on which to perform later operations. On the contrary, joins are very computation-intense and will be executed as late as possible. The resulting **execution plan** consists in a **directed acyclic graph** (DAG) that contains the tree of all required actions for a specific computation, ordered in the most effective fashion.

**This has also drawbacks.** Since the computation is optimized for the end result, the intermediate stages are discarded by default. So if you need a DataFrame multiple times, you have to cache it in memory because if you don't Spark will recompute it every single time.

### Let's practice !
How many rows have your DataFrame ?


Display columns names and then the schema

Display 10 rows of the dataframe

Sample `df_tweet` and keep only 10% of it.
  - Create a new DataFrame named `df_tweet_sampled`.
  - If computations take too long on the full DataFrame, use this one instead or add a sample transformation in your expression.

Define a DataFrame `tweet_author_hashtags` with only the `auteur` and `hashtags` columns.  
Then display 5 rows



Print 5 lines of a **df_tweet** with only the `auteur`, `mentions`, and `urls` columns.
  - `mentions` and `urls` are both nested columns in `entities`


- Filter **df_tweet** and keep only tweets with more than 1 like.  
- Display only `auteur`, `contenu` and `like_count`  
- Print 10 lines

## Basic DataFrame column manipulation

You can add/update/rename column of a DataFrame using **spark** :

- Drop : `df.drop(columnName : str )`
- Rename : `df.withColumnRenamed(oldName : str, newName : str)`
- Add/update : `df.withColumn(columnName : str, columnExpression)`

For example

```{.python}
df_tweet\
  .withColumn(                                        # computes new variable
    "like_rt_ratio",                                  # like_rt_ratio "OVERCONFIDENCE"
    df_tweet.like_count /df_tweet.retweet_count)
```

See [here](https://spark.apache.org/docs/3.1.1/api/python/reference/pyspark.sql.html#functions){.external target="_blank"} for the list of all functions available in an expression.


### Hands on column manipulation

Define a DataFrame with a column names `interaction_count` named `df_tweet_interaction_count`
  - This column is the sum of `like_count`, `reply_count` and `retweet_count`.


Update the DataFrame you imported at the beginning of this lab and drop the `other` column

## Advance DataFrame column manipulation
### Array manipulation

Some columns often contain arrays (lists) of values instead of just one value. This may seem surprising but this actually quite natural.
For instance, you may create an array of words from a text, or generate a list of random numbers for each observation, etc.

You may **create array of values** with:

- `split(text : string, delimiter : string)`, turning a text into an array of strings

You may **use array of values** with:

- `size(array : Array)`, getting the number of elements
- `array_contains(inputArray : Array, value : any)`, checking if some value appears
- `explode(array : Array)`, unnesting an array and duplicating other values. For instance if you use `explode()` over the hashtags value of this DataFrame:

  | Auteur | Contenu                             | Hashtags         |
  | ------ | ----------------------------------- | ---------------- |
  | Bob    | I love #Spark and #bigdata          | [Spark, bigdata] |
  | Alice  | Just finished #MHrise, best MH ever | [MHrise]         |

  You will get :

  | Auteur | Contenu                             | Hashtags         | Hashtag |
  | ------ | ----------------------------------- | ---------------- | ------- |
  | Bob    | I love #Spark and #bigdata          | [Spark, bigdata] | Spark   |
  | Bob    | I love #Spark and #bigdata          | [Spark, bigdata] | bigdata |
  | Alice  | Just finished #MHrise, best MH ever | [MHrise]         | MHrise  |



All this functions must be imported first :

```{.python}
from pyspark.sql.functions import split, explode, size, array_contains
```

Do not forget, to create a new column, you should use `withColumn()`. For example :

```{.python}
df.withColumn("new column", explode("array"))
```
####  Array manipulation, it's your turn 

Keep all the tweets with hashtags and for each remaining line, split the hashtag text into an array of hashtags


 Create a new column with the number of words of the `contenu` column. (Use `split()` + `size()`)

 Count how many tweet contain the `Ukraine` hashtag (use the `count()` action)

## User defined function


For more very specific column manipulation you will need Spark's `udf()` function (*User Defined Function*). It can be useful if Spark does not provide a feature you want. But Spark is a popular and active project, so before coding an udf, go check the documentation. For instance for natural language processing, Spark already has some [functions](https://spark.apache.org/docs/3.1.1/api/python/reference/api/pyspark.ml.feature.Tokenizer.html#pyspark.ml.feature.Tokenizer). Last things, python udf can lead to performance issues (see https://stackoverflow.com/a/38297050) and learning a little bit of scala or java can be a good idea.

For example :

```{.python}
# !!!! DOES NOT WORK !!!!
def to_lower_case(string):
	return string.lower()

df.withColumn("tweet_lower_case", to_lower_case(df.contenu))
```

will just crash. Keep in mind that Spark is a distributed system, and that Python is only installed on the central node, as a convenience to let you execute instructions on the executor nodes. But by default, pure Python functions can only be executed where Python is installed! We need `udf()` to enable Spark to send Python instructions to the worker nodes.

Let us see how it is done :

```{.python}
# imports
from pyspark.sql.functions import udf
from pyspark.sql.functions import explode
from pyspark.sql.types import StringType

# pure python functions
def to_lower_case(string):
    return string.lower()

# user defined function(we use a lambda function to create the udf)
to_lower_case_udf = udf(
    lambda x: to_lower_case(x), StringType()
)

# df manipulation
df_tweet_small\
  .select("auteur","hashtags")\
  .filter("size(hashtags)!=0")\
  .withColumn("hashtag", explode("hashtags"))\
  .withColumn("hashtag", to_lower_case_udf("hashtag")).show(10)
```

---

#### User defined function

Create an user defined function that counts how many words a tweet contains.
  - your function will return an `IntegerType` and not a `StringType`



## Aggregation functions


Spark offer a variety of aggregation functions :

- `count(column : string)` will count every not null value of the specify column. You cant use `count(1)` of `count("*")` to count every line (even row with only null values)

- `countDisctinct(column : string)` and `approx_count_distinct(column : string, percent_error: float)`. If the exact number is irrelevant, `approx_count_distinct()`should be preferred.

  Counting distinct elements cannot be done in parallel, and need a lot data transfer. But if you only need an approximation, there is a algorithm, named hyper-log-log (more info [here](https://databricks.com/fr/blog/2016/05/19/approximate-algorithms-in-apache-spark-hyperloglog-and-quantiles.html)) that can be parallelized.

  ```{.python}
  from pyspark.sql.functions import count, countDistinct, approx_count_distinct

  df.select(count("col1")).show()
  df.select(countDistinct("col1")).show()
  df.select(approx_count_distinct("col1"), 0.1).show()
  ```

- You have access to all other common functions `min()`, `max()`, `first()`, `last()`, `sum()`, `sumDistinct()`, `avg()` etc (you should import them first `from pyspark.sql.functions import min, max, avg, first, last, sum, sumDistinct`)

### Your turn 

What are the min, max, average of `interaction_count` (use `df_tweet_interaction_count` created earlier)
  - don't forget to import the required functions


[ ] How many tweets have hashtags ?  
[ ] Distinct hashtags ?  
[ ] Try the approximative count with 0.1 and 0.01 as maximum estimation error allowed.


## Grouping functions

Like SQL you can group row by a criteria with Spark. Just use the `groupBy(column : string)` method. Then you can compute some aggregation over those groups.

```{.python}
df\
  .groupBy("col1")\
  .agg(count("col2").alias("quantity"))\           # alias is use to specify the name of the new column
  .show()
```

The `agg()` method can take multiples argument to compute multiple aggregation at once.

```{.python}
df\
  .groupBy("col1")\
  .agg(count("col2").alias("quantity"),
       min("col2").alias("min"),
       avg("col3").alias("avg3"))\
  .show()
```

Aggregation and grouping transformations work differently than the previous method like `filter()`, `select()`, `withColumn()` etc. Those transformations cannot be run over each partitions in parallel, and need to transfer data between partitions and executors.  They are called "wide transformations"

![](img/spark_exemple2_pipeline.png)

### ✍Hands-on - Grouping functions
[ ] Compute a daframe with the min, max and average retweet of each `auteur`.  
[ ] Then order it by the max number of retweet in descending order by .


## Spark SQL

Spark understand SQL statement. It's not a hack nor a workaround to use SQL in Spark, it's one a the most powerful feature in Spark. To use SQL you will need :

1. Register a view pointing to your DataFrame

   ```{.python}
   my_df.createOrReplaceTempView(viewName : str)
   ```

2. Use the sql function

   ```{.python}
   spark.sql("""
   Your SQL statement
   """)
   ```

   You could manipulate every registered DataFrame by their view name with plain SQL.

In fact you can do most of this tutorial without any knowledge in PySpark nor Spark.
Many things can only be done in Spark if you know SQL and how to use it in Spark.

### ✍Hands-on - Spark SQL

[ ] How many tweets have hashtags ?  
[ ] Distinct hashtags ?


 Compute a dataframe with the min, max and average retweet of each `auteur` using Spark SQL

### New feature (spark 4)

The pipe syntax was introduced for spark SQL: pipe operator |> for SQL queries is similar to the Unix pipe.
The |> operator allows you to pipe the results of one query into another as a table reference:

-- Traditional approach with CTEs
```sql
WITH filtered AS (
  SELECT * FROM users WHERE age > 18
),
aggregated AS (
  SELECT country, COUNT(*) as count FROM filtered GROUP BY country
)
SELECT * FROM aggregated ORDER BY count DESC;
```

-- Using pipe operator
```sql
TABLE users
  |> SELECT * WHERE age > 18
  |> SELECT country, COUNT(*) as count GROUP BY country
  |> SELECT * ORDER BY count DESC;
```


Starting with TABLE:
```sql
TABLE my_table
  |> SELECT col1, col2 WHERE condition
  |> SELECT col1, aggregate_func(col2) GROUP BY col1;
```

Or starting with a SELECT:

```sql
SELECT * FROM my_table WHERE status = 'active'
  |> SELECT category, SUM(amount) as total GROUP BY category
  |> SELECT * WHERE total > 1000;
```

This can make SQL queries more readable by showing the data flow from left to right (or top to bottom), rather than nesting subqueries or using multiple CTEs. It's particularly useful for exploratory data analysis in SQL notebooks.


## What about joins ? 

Create two dataframes to join and then, join them. Find several ways to do it, else it's not fun.

### Did you notice any differences in time execution ?

None! Why ? Because the code goes throught the catalyst optimizer whatever way you write it and module you use. 
Here is a small pictures of what's happening under the hood. 


![](img/Catalyst-Optimizer-diagram.png)

Hence, you'll have understood by now, using spark sql or not is a matter of personnal and team preferences. It also depends on the existing, if exists, codebase and long term intentions. For example it can be easier to move out of spark if you use SQL queries and intend to use any SQL compatible framework. (ex: what if we performed the last join using duckdb ?)



### What happens when you perform joins on small dataframes or on big dataframe ? Let's dive into joins 


![](img/broadcast_join.PNG)
![](img/join_bigtable.PNG)



- Between two tables with high data volume

Spark performs a shuffle join (a wide transformation), meaning that each node communicates with every other node to exchange data based on the join keys.
This implies that all workers communicate with one another throughout the entire join process if the data is not properly partitioned. As a result, this takes a significant amount of time, especially since the network can become congested due to the traffic.
In the case of well-partitioned data, we can observe a narrow transformation and thus avoid having all nodes communicate with each other.

- Join between tables with low and high data volume

To optimize this join operation, when possible, it is beneficial to use a Broadcast Join. Indeed, when one of the joined tables is small enough in size, it can be duplicated in memory across all worker nodes; this is known as broadcasting.

The second table to be joined, which has a large data volume, is split into partitions distributed across the nodes of the cluster. Thus, by duplicating the smaller table across all nodes, the join no longer requires data exchange within the cluster (except for the broadcast of the low-volume table itself).

Broadcasting therefore helps improve the execution speed of the join.

**Note**: Spark can automatically choose to perform a broadcast join, or this behavior can be explicitly enforced by calling the broadcast function (join(broadcast(myTable))).

To improve Spark’s default behavior, it is possible to modify the configuration—particularly by increasing the allocated space if sufficient memory resources are available—using the spark.sql.autoBroadcastHashJoin flag. Its default value is 10 MB. Thus, a broadcast hash join is selected when one of the tables has a size smaller than 10 MB. It is also possible to force broadcasting by setting the parameter spark.sql.autoBroadcastHashJoin to -1.

- Join between two low-volume tables

In the case of a join between two small tables, it is preferable to let Spark decide on the execution strategy. However, it is still possible to use a broadcast join if anomalies are observed.

Be cautious before performing a broadcast join: ensure that the size of the DataFrame is smaller than the available driver memory (preferably less than 75% of the available space).

## Optimizations ? 

### Is spark really necessary for your use case ?
If you do not have a lot of data, it can be really costly to use spark instead of another solution like duckdb. If you wish for an order of magnitude, until 500Go duckdb performs really well. After that you'll have to think of your use case are you aggregating data ? If so, you might still prefer something else than spark. As a demonstration, because we did not work on a lot of data, you can try to execute the above code with duckdb and compare the execution time. 

### Partition your data



![](img/narrow_wide_transformation.png)



### Skewness

Data skewness in Spark is a critical performance issue that occurs when data is distributed unevenly across partitions, causing some tasks to take much longer than others. 

It happens when:  

- Partition imbalance: Some partitions have significantly more data than others
- Key skewness: Certain keys in join/groupBy operations have disproportionately more records
Result: A few tasks process most of the data while others finish quickly, wasting cluster resources

Solution : 

- choose a correct key to partition the data,
- salting : Add random prefixes to skewed keys to distribute them across partitions (Useful for joins and aggregations)
- broadcast Joins : broadcast smaller tables to avoid shuffling large skewed datasets (Use spark.sql.autoBroadcastJoinThreshold)
- adaptive query execution (AQE) : Spark 3.0+ feature that automatically handles skew (Enable with spark.sql.adaptive.enabled=true and spark.sql.adaptive.skewJoin.enabled=true)
- increase parallelism ? More partitions can help distribute skewed data better (Use repartition() or increase spark.sql.shuffle.partitions) be carefull, too many partitions can lead o poor performance
- Custom Partitioning, for example, use range partitioning instead of hash partitioning when appropriate



## End of the TP, STOP YOUR SESSION please

In [16]:
spark.stop()

### Wait
Wait a little before leaving, how long did it take to write the data of the last join in s3? Even without this question, I launched a batch and I do not have the expected result. My session stoped. How do I know what went wrong ? 

🤡 Do we restart a sparksession and redo all the above steps manually ? 🫠 Do I never stop my session even if I do not use the ressources ?
It's not practical at all and neither are a good practice.

Hopefully, we have a tool to keep trace of everything happening within our sparksession : spark history server. It stores everything you can see in the spark UI.